# Dataset Preparation: The Envision Triadic Design Dataset

**Thursday Masking Day — Tilburg Multiscale Summerschool 2026**

This notebook prepares video data from the Edelman (2011) *Envision* study for privacy-preserving masking with MaskAnyone.

### About the dataset

Edelman (2011) recorded 14 triadic teams (3 participants each) performing a 30-minute redesign task in Stanford's Center for Design Research Design Observatory. Each session was captured with **4 HD video cameras**:

| Stream | Description |
|--------|-------------|
| `cam_p1` | Camera focused on Participant 1 |
| `cam_p2` | Camera focused on Participant 2 |
| `cam_p3` | Camera focused on Participant 3 |
| `cam_overhead` | Top-down camera looking down at the shared table |

The 4 streams were composited into a single **2×2 four-up video file** using FinalCut Pro, with timecode burned into the lower-right corner.

**Session timeline per group:**
- `00:00–00:05` — Setup, briefing, KAI questionnaire  
- `00:05–00:35` — **30-minute redesign task** ← *primary masking target*  
- `00:35–01:00+` — Semi-structured exit interview

**Reference:** Edelman, J. A. (2011). *Understanding Radical Breaks: Media and Behavior in Small Teams Engaged in Redesign Scenarios.* PhD dissertation, Stanford University.


## Setup

In [ ]:
import subprocess
import json
import os
import csv
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Paths — edit DATA_ROOT to point to your copy of the Envision videos
REPO_ROOT   = Path("../")                        # thursday-masking/
DATA_ROOT   = REPO_ROOT / "data" / "raw"          # raw four-up composites go here
SPLIT_DIR   = REPO_ROOT / "data" / "split"        # individual camera streams
TRIMMED_DIR = REPO_ROOT / "data" / "trimmed"      # trimmed to task window
CLIPS_DIR   = REPO_ROOT / "data" / "workshop_clips"  # short excerpts for exercises
SAMPLE_VIDEO = REPO_ROOT / "__design.mp4"         # reference clip bundled with repo

for d in [DATA_ROOT, SPLIT_DIR, TRIMMED_DIR, CLIPS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

## Step 1: Inspect a video

Before processing, understand what you have. We use `ffprobe` to read the container metadata.

In [ ]:
def probe_video(path: Path) -> dict:
    """Return ffprobe metadata as a dict."""
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_streams", "-show_format",
        str(path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return json.loads(result.stdout)

def print_video_summary(path: Path):
    meta = probe_video(path)
    fmt  = meta["format"]
    vid  = next(s for s in meta["streams"] if s["codec_type"] == "video")
    aud  = next((s for s in meta["streams"] if s["codec_type"] == "audio"), None)

    duration_min = float(fmt["duration"]) / 60
    print(f"File     : {Path(path).name}")
    print(f"Duration : {duration_min:.1f} min ({float(fmt['duration']):.1f} s)")
    print(f"Size     : {int(fmt['size']) / 1e6:.1f} MB")
    print(f"Video    : {vid['codec_name'].upper()}  {vid['width']}×{vid['height']}  "
          f"{eval(vid['r_frame_rate']):.1f} fps")
    if aud:
        print(f"Audio    : {aud['codec_name'].upper()}  {aud.get('sample_rate','?')} Hz  "
              f"{aud.get('channels','?')} ch")
    else:
        print("Audio    : none")

print_video_summary(SAMPLE_VIDEO)

## Step 2: Visualise the 4-up composite layout

Each Envision group video is a **2×2 grid** of four synchronised camera streams.
We extract a single frame and mark the quadrant boundaries.

In [ ]:
def extract_frame(video_path: Path, time_s: float = 5.0) -> np.ndarray:
    """Extract one frame at time_s seconds (BGR → RGB)."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(time_s * fps))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"Could not read frame at {time_s}s from {video_path}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

def show_quadrant_grid(frame: np.ndarray, labels=None):
    """Draw the 2×2 stream boundaries on a composite frame."""
    h, w = frame.shape[:2]
    cx, cy = w // 2, h // 2
    default_labels = ["Participant 1", "Participant 2", "Participant 3", "Overhead"]
    labels = labels or default_labels

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(frame)

    # grid lines
    ax.axvline(cx, color="yellow", linewidth=2, linestyle="--")
    ax.axhline(cy, color="yellow", linewidth=2, linestyle="--")

    # quadrant labels
    positions = [(cx // 2, cy // 2), (cx + cx // 2, cy // 2),
                 (cx // 2, cy + cy // 2), (cx + cx // 2, cy + cy // 2)]
    for (x, y), lbl in zip(positions, labels):
        ax.text(x, y, lbl, ha="center", va="center",
                color="white", fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.6))

    ax.set_title("Four-Up Composite — stream grid", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

frame = extract_frame(SAMPLE_VIDEO, time_s=5.0)
show_quadrant_grid(frame)

> **Adjust the labels above** if your composite has a different quadrant assignment  
> (e.g. overhead in top-left). Check Figure 15 in Edelman (2011) as a reference.

## Step 3: Split composite into individual streams

We use `ffmpeg`'s `crop` filter to extract each quadrant as a separate file.  
Syntax: `crop=out_w:out_h:x:y`

In [ ]:
STREAM_LAYOUT = [
    # (label,        x_offset,  y_offset)
    ("cam_p1",       "0",       "0"),
    ("cam_p2",       "iw/2",    "0"),
    ("cam_p3",       "0",       "ih/2"),
    ("cam_overhead", "iw/2",    "ih/2"),
]

def split_composite(input_path: Path, out_dir: Path, layout=STREAM_LAYOUT,
                    dry_run: bool = False) -> list:
    """
    Split a four-up composite into individual stream files.
    Returns list of output paths.
    """
    stem = input_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)
    outputs = []

    for label, x, y in layout:
        out_path = out_dir / f"{stem}__{label}.mp4"
        crop_filter = f"crop=iw/2:ih/2:{x}:{y}"
        cmd = [
            "ffmpeg", "-y",
            "-i", str(input_path),
            "-vf", crop_filter,
            "-c:v", "libx264", "-crf", "18", "-preset", "fast",
            "-c:a", "copy",
            str(out_path)
        ]
        if dry_run:
            print("DRY RUN:", " ".join(cmd))
        else:
            print(f"Splitting → {out_path.name} ...", end=" ", flush=True)
            subprocess.run(cmd, capture_output=True, check=True)
            print("done.")
        outputs.append(out_path)

    return outputs

# Preview the commands without running them
split_composite(SAMPLE_VIDEO, SPLIT_DIR / "sample", dry_run=True)

In [ ]:
# Run on the sample video (takes a few seconds)
split_paths = split_composite(SAMPLE_VIDEO, SPLIT_DIR / "sample", dry_run=False)
for p in split_paths:
    print_video_summary(p)
    print()

### Process all group videos

Place the full Envision composite files in `data/raw/` following the naming convention  
`group_{N:02d}.mp4` (e.g. `group_02.mp4`, `group_07.mp4`, ...).

In [ ]:
def split_all_groups(data_root: Path = DATA_ROOT, split_root: Path = SPLIT_DIR):
    composites = sorted(data_root.glob("group_*.mp4"))
    if not composites:
        print(f"No group_*.mp4 files found in {data_root}. "
              "Copy the Envision videos there first.")
        return []

    all_outputs = []
    for composite in composites:
        group_id = composite.stem          # e.g. "group_07"
        out_dir  = split_root / group_id
        print(f"\n=== {group_id} ===")
        outputs = split_composite(composite, out_dir)
        all_outputs.extend(outputs)

    print(f"\nTotal streams created: {len(all_outputs)}")
    return all_outputs

# split_all_groups()  # uncomment when full videos are in data/raw/

## Step 4: Trim to the 30-minute task window

The redesign task runs from approximately **00:05** to **00:35** (5 min setup → 35 min).  
Adjust `TASK_START` and `TASK_END` per group if you have precise timecodes from VCode.

In [ ]:
# Default task window (seconds). Override per group in GROUP_OFFSETS below.
TASK_START_DEFAULT = 5 * 60    # 5:00
TASK_END_DEFAULT   = 35 * 60   # 35:00

# Per-group overrides — fill these in from your VCode timecodes
GROUP_OFFSETS = {
    # "group_02": (300, 2100),
    # "group_07": (310, 2110),
}

def trim_stream(input_path: Path, out_dir: Path,
                start_s: float, end_s: float,
                dry_run: bool = False) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / input_path.name
    cmd = [
        "ffmpeg", "-y",
        "-ss", str(start_s),
        "-to", str(end_s),
        "-i", str(input_path),
        "-c", "copy",
        "-avoid_negative_ts", "make_zero",
        str(out_path)
    ]
    if dry_run:
        print("DRY RUN:", " ".join(cmd))
    else:
        subprocess.run(cmd, capture_output=True, check=True)
    return out_path

def trim_all_groups(split_root: Path = SPLIT_DIR, trimmed_root: Path = TRIMMED_DIR):
    all_trimmed = []
    for group_dir in sorted(split_root.iterdir()):
        if not group_dir.is_dir():
            continue
        group_id = group_dir.name
        start_s, end_s = GROUP_OFFSETS.get(group_id,
                                            (TASK_START_DEFAULT, TASK_END_DEFAULT))
        out_dir = trimmed_root / group_id
        print(f"\n=== {group_id}  [{start_s/60:.1f}–{end_s/60:.1f} min] ===")
        for stream_file in sorted(group_dir.glob("*.mp4")):
            p = trim_stream(stream_file, out_dir, start_s, end_s)
            print(f"  Trimmed: {p.name}")
            all_trimmed.append(p)
    return all_trimmed

# trim_all_groups()  # uncomment after splitting

## Step 5: MaskAnyone compatibility check

MaskAnyone expects **H.264 MP4** at a resolution ≤ 1080p.  
If your source is VP9 or ProRes, re-encode it here.

In [ ]:
def check_maskanyone_compat(path: Path) -> dict:
    meta  = probe_video(path)
    vid   = next(s for s in meta["streams"] if s["codec_type"] == "video")
    codec = vid["codec_name"]
    w, h  = vid["width"], vid["height"]
    issues = []
    if codec not in ("h264", "avc"):
        issues.append(f"codec is {codec.upper()} — needs H.264")
    if max(w, h) > 1920:
        issues.append(f"resolution {w}×{h} exceeds 1920 on longest side")
    return {"path": path.name, "codec": codec, "resolution": f"{w}×{h}",
            "ok": len(issues) == 0, "issues": issues}

def reencode_for_maskanyone(input_path: Path, output_path: Path,
                             target_fps: int = 25) -> Path:
    """Re-encode to H.264 MP4, optionally downsampling to target_fps."""
    cmd = [
        "ffmpeg", "-y",
        "-i", str(input_path),
        "-r", str(target_fps),
        "-c:v", "libx264", "-crf", "23", "-preset", "medium",
        "-c:a", "aac", "-b:a", "128k",
        str(output_path)
    ]
    subprocess.run(cmd, capture_output=True, check=True)
    return output_path

# Check the sample
result = check_maskanyone_compat(SAMPLE_VIDEO)
print(result)

if not result["ok"]:
    print("\nRe-encoding sample for MaskAnyone compatibility...")
    fixed = SAMPLE_VIDEO.parent / (SAMPLE_VIDEO.stem + "_h264.mp4")
    reencode_for_maskanyone(SAMPLE_VIDEO, fixed)
    print(f"Saved to: {fixed}")
    print(check_maskanyone_compat(fixed))

## Step 6: Create workshop clips

For the hands-on exercises we cut **5-minute excerpts** from each participant stream.  
These are short enough to process quickly on the SYNAPSIS server during the session.

In [ ]:
CLIP_DURATION_S = 5 * 60   # 5 minutes

# Interesting segments from the Edelman case selection (Table 1 in dissertation)
# Groups 2, 7, 9, 10 were selected for deep coding — good candidates for clips
WORKSHOP_SEGMENTS = {
    "group_02": 0,    # seconds into the task window (start of clip)
    "group_07": 0,
    "group_09": 0,
    "group_10": 0,
}

def make_workshop_clips(trimmed_root: Path = TRIMMED_DIR,
                         clips_root: Path   = CLIPS_DIR,
                         segments: dict     = WORKSHOP_SEGMENTS,
                         streams_to_include = ("cam_p1", "cam_p2", "cam_p3")):
    """
    Cut one short clip per participant stream for selected groups.
    Overhead camera is excluded by default (no identity to mask).
    """
    clips_root.mkdir(parents=True, exist_ok=True)
    clip_manifest = []

    for group_id, offset_s in segments.items():
        group_dir = trimmed_root / group_id
        if not group_dir.exists():
            print(f"  {group_id}: no trimmed streams found, skipping.")
            continue

        for stream_label in streams_to_include:
            candidates = list(group_dir.glob(f"*__{stream_label}.mp4"))
            if not candidates:
                continue
            src = candidates[0]
            out = clips_root / f"{group_id}__{stream_label}__clip.mp4"
            cmd = [
                "ffmpeg", "-y",
                "-ss", str(offset_s),
                "-t",  str(CLIP_DURATION_S),
                "-i",  str(src),
                "-c",  "copy",
                str(out)
            ]
            subprocess.run(cmd, capture_output=True, check=True)
            clip_manifest.append({"group": group_id, "stream": stream_label,
                                   "file": out.name, "offset_s": offset_s,
                                   "duration_s": CLIP_DURATION_S})
            print(f"  {out.name}")

    # Write manifest CSV
    manifest_path = clips_root / "manifest.csv"
    if clip_manifest:
        with open(manifest_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=clip_manifest[0].keys())
            writer.writeheader()
            writer.writerows(clip_manifest)
        print(f"\nManifest written to {manifest_path}")

    return clip_manifest

# make_workshop_clips()  # uncomment after trimming

## Step 7: Inventory all prepared files

Generate a summary CSV of every processed video: group, stream, duration, resolution, codec.

In [ ]:
def inventory(root: Path, output_csv: Path = None) -> list:
    rows = []
    for mp4 in sorted(root.rglob("*.mp4")):
        meta = probe_video(mp4)
        fmt  = meta["format"]
        vid  = next((s for s in meta["streams"] if s["codec_type"] == "video"), {})
        rows.append({
            "path":       str(mp4.relative_to(root)),
            "duration_s": round(float(fmt.get("duration", 0)), 1),
            "size_mb":    round(int(fmt.get("size", 0)) / 1e6, 1),
            "codec":      vid.get("codec_name", "?"),
            "width":      vid.get("width", "?"),
            "height":     vid.get("height", "?"),
            "fps":        round(eval(vid.get("r_frame_rate", "0/1")), 2),
        })

    if output_csv and rows:
        with open(output_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
        print(f"Inventory saved to {output_csv}")

    return rows

# Show inventory of trimmed streams once they exist
rows = inventory(REPO_ROOT, output_csv=REPO_ROOT / "data" / "inventory.csv")
for r in rows:
    print(r)

## Summary: full pipeline

```
data/raw/group_02.mp4          ← four-up composite from Edelman dataset
         group_07.mp4
         group_09.mp4
         group_10.mp4
             │
             ▼  split_all_groups()
data/split/group_02/group_02__cam_p1.mp4
                    group_02__cam_p2.mp4
                    group_02__cam_p3.mp4
                    group_02__cam_overhead.mp4
             │
             ▼  trim_all_groups()
data/trimmed/group_02/group_02__cam_p1.mp4   ← 30 min task window only
             │
             ▼  make_workshop_clips()
data/workshop_clips/group_02__cam_p1__clip.mp4  ← 5 min excerpt
                    manifest.csv
             │
             ▼  MaskAnyone (SYNAPSIS server)
masked/group_02__cam_p1__clip__masked.mp4
```

**Which streams go into MaskAnyone?**  
- `cam_p1`, `cam_p2`, `cam_p3` — identity masking of individual participants  
- `cam_overhead` — optional; top-down view may show less face but reveals hands and
  sketching gestures (relevant for the Wednesday pose estimation connection)

**Connection to the rest of Thursday:**  
The clips you create here flow directly into the MaskAnyone hands-on sessions  
(11:30–12:30 and 13:30–14:30) and MaskBench evaluation (14:30–15:15).
